# 01 — Occlusion stress-testing (SAM)

Controlled occlusion samples from real driving frames.

**Outputs:** `outputs/SAM3/research/occlusion_augmentation/`

Aggregate metrics already saved: `summary.json`, `summary_iou.png`.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "sam3_augmentation":
    ROOT = ROOT.parents[1]

BDD_IMG = ROOT / "Research_Data/BDD100K/hf_dgural_bdd100k/data"
OUT_DIR = ROOT / "outputs/SAM3/research/occlusion_augmentation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CONCEPT = "pedestrian"
OCCLUDER_CONCEPT = "car"
OCCLUSION_RATIO = 0.5
FRAME_IDX = 0

def pick_device() -> str:
    override = os.environ.get("SAM3_DEVICE", "").strip().lower()
    if override in {"cpu", "mps", "cuda"}:
        return override
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

DEVICE = pick_device()
frame_paths = sorted(BDD_IMG.glob("*.jpg"))

print(f"ROOT={ROOT}")
print(f"BDD_IMG exists={BDD_IMG.is_dir()}  n_images={len(frame_paths)}")
print(f"OUT_DIR={OUT_DIR}")
print(f"DEVICE={DEVICE}")
print(f"target={TARGET_CONCEPT!r}  occluder={OCCLUDER_CONCEPT!r}  ratio={OCCLUSION_RATIO}")

In [ ]:
FRAME_IDX = 7269 # change this until you like the scene

assert frame_paths, "No BDD images — re-run Cell 0"
assert 0 <= FRAME_IDX < len(frame_paths), f"FRAME_IDX out of range 0..{len(frame_paths)-1}"

frame_path = frame_paths[FRAME_IDX]
image = Image.open(frame_path).convert("RGB")
image_np = np.asarray(image)

print(f"idx={FRAME_IDX}  file={frame_path.name}  shape={image_np.shape}")

plt.figure(figsize=(12, 5))
plt.imshow(image_np)
plt.title(f"BDD {frame_path.name}")
plt.axis("off")
plt.show()

In [ ]:
from sam3.model.device_utils import install_cuda_compat_shim
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

CONF = 0.3
device = install_cuda_compat_shim()
print(f"building SAM3 on {device}...")

model = build_sam3_image_model(device=device, load_from_HF=True)
processor = Sam3Processor(model, device=device, confidence_threshold=CONF)

def masks_to_union(masks) -> np.ndarray:
    h, w = image_np.shape[:2]
    if masks is None or len(masks) == 0:
        return np.zeros((h, w), dtype=bool)
    m = masks.detach().cpu().numpy()
    if m.ndim == 4:
        m = m[:, 0]
    union = np.zeros((h, w), dtype=bool)
    for i in range(m.shape[0]):
        mi = m[i] > 0.5
        if mi.shape != (h, w):
            mi = (
                np.array(
                    Image.fromarray((mi.astype(np.uint8) * 255)).resize(
                        (w, h), Image.NEAREST
                    )
                )
                > 127
            )
        union |= mi
    return union

state = processor.set_image(image)

print(f"prompting {TARGET_CONCEPT!r}...")
out_t = processor.set_text_prompt(state=state, prompt=TARGET_CONCEPT)
target_mask = masks_to_union(out_t["masks"])
print(f"  instances={len(out_t['masks'])}  pixels={int(target_mask.sum())}")

print(f"prompting {OCCLUDER_CONCEPT!r}...")
out_o = processor.set_text_prompt(state=state, prompt=OCCLUDER_CONCEPT)
occluder_mask = masks_to_union(out_o["masks"])
print(f"  instances={len(out_o['masks'])}  pixels={int(occluder_mask.sum())}")

vis = image_np.copy().astype(np.float32)
vis[target_mask] = vis[target_mask] * 0.45 + np.array([0, 220, 80]) * 0.55
vis[occluder_mask] = vis[occluder_mask] * 0.45 + np.array([220, 80, 0]) * 0.55

plt.figure(figsize=(12, 5))
plt.imshow(vis.astype(np.uint8))
plt.title(f"green={TARGET_CONCEPT}  orange={OCCLUDER_CONCEPT}")
plt.axis("off")
plt.show()

assert target_mask.any(), f"No {TARGET_CONCEPT} — try another FRAME_IDX"
assert occluder_mask.any(), f"No {OCCLUDER_CONCEPT} — try another FRAME_IDX"

In [ ]:
from matplotlib.colors import ListedColormap

def _overlay(rgb, mask, color, alpha=0.55):
    out = rgb.astype(np.float32).copy()
    c = np.asarray(color, dtype=np.float32)
    out[mask] = out[mask] * (1 - alpha) + c * alpha
    return np.clip(out, 0, 255).astype(np.uint8)

TARGET_COLOR = (0, 200, 90)      # green
OCCLUDER_COLOR = (230, 90, 40)   # orange

t_pix = int(target_mask.sum())
o_pix = int(occluder_mask.sum())
overlap = target_mask & occluder_mask

# Panel figure
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(
    f"BDD {frame_path.name} | idx={FRAME_IDX} | "
    f"{TARGET_CONCEPT}={t_pix}px  {OCCLUDER_CONCEPT}={o_pix}px  overlap={int(overlap.sum())}px",
    fontsize=13,
)

# 1 raw
axes[0, 0].imshow(image_np)
axes[0, 0].set_title("Original")

# 2 target only
axes[0, 1].imshow(_overlay(image_np, target_mask, TARGET_COLOR))
axes[0, 1].set_title(f"Target: {TARGET_CONCEPT}")

# 3 occluder only
axes[0, 2].imshow(_overlay(image_np, occluder_mask, OCCLUDER_COLOR))
axes[0, 2].set_title(f"Occluder: {OCCLUDER_CONCEPT}")

# 4 both overlays
both = _overlay(image_np, target_mask, TARGET_COLOR)
both = _overlay(both, occluder_mask, OCCLUDER_COLOR)
axes[1, 0].imshow(both)
axes[1, 0].set_title("Both overlays")

# 5 binary masks (clean GT view)
stack = np.zeros((*target_mask.shape, 3), dtype=np.uint8)
stack[target_mask] = TARGET_COLOR
stack[occluder_mask] = OCCLUDER_COLOR
stack[overlap] = (255, 255, 0)  # yellow = already overlapping in scene
axes[1, 1].imshow(stack)
axes[1, 1].set_title("Mask map (yellow=overlap)")

# 6 coverage bars
axes[1, 2].bar(
    [TARGET_CONCEPT, OCCLUDER_CONCEPT, "overlap"],
    [t_pix, o_pix, int(overlap.sum())],
    color=["#00c85a", "#e65a28", "#e6e600"],
)
axes[1, 2].set_ylabel("pixels")
axes[1, 2].set_title("Mask area")
axes[1, 2].tick_params(axis="x", rotation=15)

for ax in axes.ravel()[:5]:
    ax.axis("off")
plt.tight_layout()
plt.show()

# Optional: save a review sheet
review_path = OUT_DIR / f"review_{FRAME_IDX}_{frame_path.stem}.png"
fig.savefig(review_path, dpi=150, bbox_inches="tight")
print(f"saved {review_path}")

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "notebooks/sam3_augmentation"))

from occlusion_synth import run_from_processor, render_qc_figure

# needs: processor, image, image_np, FRAME_IDX, frame_path, TARGET_CONCEPT, OCCLUDER_CONCEPT, OCCLUSION_RATIO, OUT_DIR

occ = run_from_processor(
    processor,
    image,
    image_np,
    target_concept=TARGET_CONCEPT,
    occluder_concept=OCCLUDER_CONCEPT,
    ratio=OCCLUSION_RATIO,
)

print(
    f"ratio={occ['ratio_achieved']:.3f}  "
    f"ped_hw={occ['target_bbox_hw']}  car_hw={occ['occluder_bbox_hw']}  "
    f"visible={int(occ['target_visible'].sum())}  hidden={int(occ['target_occluded'].sum())}"
)

render_qc_figure(
    image_np,
    occ,
    frame_path.name,
    out_path=OUT_DIR / f"occlusion_{FRAME_IDX}_{frame_path.stem}.png",
)

In [ ]:
import json
from PIL import Image as PILImage

sample_dir = OUT_DIR / f"sample_{FRAME_IDX}_{frame_path.stem}"
sample_dir.mkdir(parents=True, exist_ok=True)

def save_mask(mask: np.ndarray, path: Path) -> None:
    PILImage.fromarray((mask.astype(np.uint8) * 255)).save(path)

# RGB
PILImage.fromarray(image_np).save(sample_dir / "original.jpg", quality=95)
PILImage.fromarray(occ["hard_rgb"]).save(sample_dir / "hard.jpg", quality=95)

# GT masks
save_mask(occ["target_full"], sample_dir / "target_full.png")
save_mask(occ["target_visible"], sample_dir / "target_visible.png")
save_mask(occ["target_occluded"], sample_dir / "target_occluded.png")
save_mask(occ["occluder_src"], sample_dir / "occluder_src.png")

meta = {
    "dataset": "BDD100K",
    "source_image": frame_path.name,
    "frame_idx": int(FRAME_IDX),
    "target_concept": occ.get("target_concept", TARGET_CONCEPT),
    "occluder_concept": occ.get("occluder_concept", OCCLUDER_CONCEPT),
    "occlusion_mode": occ.get("mode", "bottom"),
    "ratio_requested": float(occ["ratio_requested"]),
    "ratio_achieved": float(occ["ratio_achieved"]),
    "target_area_px": int(occ["target_full"].sum()),
    "visible_area_px": int(occ["target_visible"].sum()),
    "occluded_area_px": int(occ["target_occluded"].sum()),
    "target_bbox_hw": list(occ.get("target_bbox_hw", [])),
    "paths": {
        "original": "original.jpg",
        "hard": "hard.jpg",
        "target_full": "target_full.png",
        "target_visible": "target_visible.png",
        "target_occluded": "target_occluded.png",
        "occluder_src": "occluder_src.png",
    },
}

meta_path = sample_dir / "meta.json"
meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

print(f"saved sample → {sample_dir}")
print(json.dumps(meta, indent=2))

In [ ]:
def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union else 0.0


def masks_to_union(masks, h, w) -> np.ndarray:
    if masks is None or len(masks) == 0:
        return np.zeros((h, w), dtype=bool)
    m = masks.detach().cpu().numpy()
    if m.ndim == 4:
        m = m[:, 0]
    union = np.zeros((h, w), dtype=bool)
    for i in range(m.shape[0]):
        mi = m[i] > 0.5
        if mi.shape != (h, w):
            mi = (
                np.array(
                    Image.fromarray((mi.astype(np.uint8) * 255)).resize(
                        (w, h), Image.NEAREST
                    )
                )
                > 127
            )
        union |= mi
    return union


hard_img = Image.fromarray(occ["hard_rgb"])
h, w = occ["hard_rgb"].shape[:2]

# Baseline on original (clean)
state0 = processor.set_image(image)
pred_clean = masks_to_union(
    processor.set_text_prompt(state=state0, prompt=TARGET_CONCEPT)["masks"], h, w
)

# Stress test on hard image
state1 = processor.set_image(hard_img)
pred_hard = masks_to_union(
    processor.set_text_prompt(state=state1, prompt=TARGET_CONCEPT)["masks"], h, w
)

gt_full = occ["target_full"]
gt_vis = occ["target_visible"]

iou_clean_full = mask_iou(pred_clean, gt_full)
iou_hard_full = mask_iou(pred_hard, gt_full)
iou_hard_visible = mask_iou(pred_hard, gt_vis)  # main occlusion metric

print("=== occlusion stress eval ===")
print(f"clean  vs full GT IoU     = {iou_clean_full:.3f}")
print(f"hard   vs full GT IoU     = {iou_hard_full:.3f}")
print(f"hard   vs visible GT IoU  = {iou_hard_visible:.3f}  ← primary")

# ---- visuals ----
TARGET_COLOR = (0, 200, 90)
PRED_COLOR = (80, 160, 255)
HIDDEN_COLOR = (230, 80, 40)

def ov(rgb, mask, color, a=0.55):
    out = rgb.astype(np.float32).copy()
    c = np.asarray(color, dtype=np.float32)
    out[mask] = out[mask] * (1 - a) + c * a
    return np.clip(out, 0, 255).astype(np.uint8)

ys, xs = np.where(gt_full)
y0, y1 = max(0, ys.min() - 40), min(h, ys.max() + 40)
x0, x1 = max(0, xs.min() - 40), min(w, xs.max() + 40)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(
    f"SAM3 occlusion stress | {frame_path.name} | "
    f"visible-IoU={iou_hard_visible:.2f}  full-IoU clean→hard "
    f"{iou_clean_full:.2f}→{iou_hard_full:.2f}",
    fontsize=13,
)

axes[0, 0].imshow(image_np)
axes[0, 0].set_title("Original")

axes[0, 1].imshow(occ["hard_rgb"])
axes[0, 1].set_title("Hard (legs occluded)")

axes[0, 2].imshow(ov(ov(image_np, gt_vis, TARGET_COLOR), occ["target_occluded"], HIDDEN_COLOR))
axes[0, 2].set_title("GT: green=visible  orange=hidden")

axes[1, 0].imshow(ov(image_np, pred_clean, PRED_COLOR))
axes[1, 0].set_title(f"SAM3 on clean (IoU-full={iou_clean_full:.2f})")

axes[1, 1].imshow(ov(occ["hard_rgb"], pred_hard, PRED_COLOR))
axes[1, 1].set_title(f"SAM3 on hard (IoU-vis={iou_hard_visible:.2f})")

# zoom: hard | pred | GT visible
z = np.concatenate(
    [
        occ["hard_rgb"][y0:y1, x0:x1],
        ov(occ["hard_rgb"], pred_hard, PRED_COLOR)[y0:y1, x0:x1],
        ov(occ["hard_rgb"], gt_vis, TARGET_COLOR)[y0:y1, x0:x1],
    ],
    axis=1,
)
axes[1, 2].imshow(z)
axes[1, 2].set_title("Zoom: hard | pred | GT-visible")

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

eval_png = OUT_DIR / f"eval_{FRAME_IDX}_{frame_path.stem}.png"
fig.savefig(eval_png, dpi=150, bbox_inches="tight")
print(f"saved {eval_png}")

# append scores into sample meta if Cell 4 ran
eval_scores = {
    "iou_clean_vs_full": iou_clean_full,
    "iou_hard_vs_full": iou_hard_full,
    "iou_hard_vs_visible": iou_hard_visible,
}
if "sample_dir" in globals() and (sample_dir / "meta.json").exists():
    meta = json.loads((sample_dir / "meta.json").read_text())
    meta["eval"] = eval_scores
    (sample_dir / "meta.json").write_text(json.dumps(meta, indent=2))
    print("updated", sample_dir / "meta.json")
else:
    print(json.dumps(eval_scores, indent=2))

In [ ]:
def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union else 0.0


def instance_masks(masks, h, w) -> list[np.ndarray]:
    if masks is None or len(masks) == 0:
        return []
    m = masks.detach().cpu().numpy()
    if m.ndim == 4:
        m = m[:, 0]
    out = []
    for i in range(m.shape[0]):
        mi = m[i] > 0.5
        if mi.shape != (h, w):
            mi = (
                np.array(
                    Image.fromarray((mi.astype(np.uint8) * 255)).resize(
                        (w, h), Image.NEAREST
                    )
                )
                > 127
            )
        if mi.any():
            out.append(mi)
    return out


def best_instance_iou(pred_list: list[np.ndarray], gt: np.ndarray):
    if not pred_list or not gt.any():
        return 0.0, None, -1
    scores = [mask_iou(p, gt) for p in pred_list]
    k = int(np.argmax(scores))
    return float(scores[k]), pred_list[k], k


h, w = image_np.shape[:2]
gt_full = occ["target_full"]
gt_vis = occ["target_visible"]

# Clean
state0 = processor.set_image(image)
preds_clean = instance_masks(
    processor.set_text_prompt(state=state0, prompt=TARGET_CONCEPT)["masks"], h, w
)
# Hard
state1 = processor.set_image(Image.fromarray(occ["hard_rgb"]))
preds_hard = instance_masks(
    processor.set_text_prompt(state=state1, prompt=TARGET_CONCEPT)["masks"], h, w
)

iou_c_full, pred_c, idx_c = best_instance_iou(preds_clean, gt_full)
iou_h_full, pred_h, idx_h = best_instance_iou(preds_hard, gt_full)
iou_h_vis, pred_h_vis, idx_hv = best_instance_iou(preds_hard, gt_vis)

# Also show old union scores for comparison
union_c = np.logical_or.reduce(preds_clean) if preds_clean else np.zeros((h, w), bool)
union_h = np.logical_or.reduce(preds_hard) if preds_hard else np.zeros((h, w), bool)

print("=== best-instance IoU (fair) ===")
print(f"clean  vs full GT     = {iou_c_full:.3f}   (best of {len(preds_clean)} preds, idx={idx_c})")
print(f"hard   vs full GT     = {iou_h_full:.3f}   (best of {len(preds_hard)} preds, idx={idx_h})")
print(f"hard   vs visible GT  = {iou_h_vis:.3f}   ← primary  (idx={idx_hv})")
print("--- old union IoU (for contrast) ---")
print(f"union clean vs full   = {mask_iou(union_c, gt_full):.3f}")
print(f"union hard  vs full   = {mask_iou(union_h, gt_full):.3f}")
print(f"union hard  vs visible= {mask_iou(union_h, gt_vis):.3f}")

# ---- visuals ----
GT_COLOR = (0, 200, 90)
PRED_COLOR = (80, 160, 255)
HIDDEN_COLOR = (230, 80, 40)

def ov(rgb, mask, color, a=0.55):
    if mask is None:
        return rgb
    out = rgb.astype(np.float32).copy()
    c = np.asarray(color, dtype=np.float32)
    out[mask] = out[mask] * (1 - a) + c * a
    return np.clip(out, 0, 255).astype(np.uint8)

ys, xs = np.where(gt_full)
y0, y1 = max(0, ys.min() - 40), min(h, ys.max() + 40)
x0, x1 = max(0, xs.min() - 40), min(w, xs.max() + 40)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(
    f"Best-instance SAM3 eval | {frame_path.name} | "
    f"vis-IoU={iou_h_vis:.2f}  full clean→hard {iou_c_full:.2f}→{iou_h_full:.2f}",
    fontsize=13,
)

axes[0, 0].imshow(ov(image_np, gt_full, GT_COLOR))
axes[0, 0].set_title("GT primary ped (full)")

axes[0, 1].imshow(ov(image_np, pred_c, PRED_COLOR) if pred_c is not None else image_np)
axes[0, 1].set_title(f"Best pred on clean (IoU={iou_c_full:.2f})")

axes[0, 2].imshow(ov(occ["hard_rgb"], pred_h, PRED_COLOR) if pred_h is not None else occ["hard_rgb"])
axes[0, 2].set_title(f"Best pred on hard (IoU-full={iou_h_full:.2f})")

axes[1, 0].imshow(ov(ov(occ["hard_rgb"], gt_vis, GT_COLOR), occ["target_occluded"], HIDDEN_COLOR))
axes[1, 0].set_title("Hard GT: green visible / orange hidden")

axes[1, 1].imshow(ov(occ["hard_rgb"], pred_h_vis, PRED_COLOR) if pred_h_vis is not None else occ["hard_rgb"])
axes[1, 1].set_title(f"Best pred vs visible (IoU={iou_h_vis:.2f})")

z = np.concatenate(
    [
        ov(image_np, gt_full, GT_COLOR)[y0:y1, x0:x1],
        ov(occ["hard_rgb"], pred_h if pred_h is not None else np.zeros((h, w), bool), PRED_COLOR)[y0:y1, x0:x1],
        ov(occ["hard_rgb"], gt_vis, GT_COLOR)[y0:y1, x0:x1],
    ],
    axis=1,
)
axes[1, 2].imshow(z)
axes[1, 2].set_title("Zoom: GT-full | best-pred-hard | GT-visible")

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

out_png = OUT_DIR / f"eval_bestinst_{FRAME_IDX}_{frame_path.stem}.png"
fig.savefig(out_png, dpi=150, bbox_inches="tight")
print("saved", out_png)